# STT_MultiLingva — Colab runner

Transcribes a meeting that mixes Russian and English (optionally Armenian) on a
free Colab T4. The model runs on the GPU Colab gives you; nothing is sent to a
transcription API.

## Read this before uploading anything

**Colab is Google infrastructure.** Audio uploaded here leaves your machine and
lands on Google's servers. If the recording is confidential, that is the same
disclosure you were avoiding by not using a transcription API — the model is
local to the runtime, but the file is not.

Use this for audio you are free to share. For a confidential meeting, run the
Docker image on a GPU inside your own perimeter; `README.md` covers it.

**Runtime → Change runtime type → T4 GPU** before running anything.

---

Run sections 1–4, then **section 5** for the drag-and-drop interface. Section 6
is the command-line path, for batch work or scripting.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


## 2. Install

`faster-whisper` pulls CTranslate2, which needs the cuDNN 9 runtime. Colab ships
CUDA but not always the matching cuDNN, so it goes in explicitly.

In [ ]:
!pip install -q faster-whisper==1.2.1 "gradio==6.*"
!pip install -q nvidia-cudnn-cu12==9.*
import os, pathlib, nvidia.cudnn
# CTranslate2 loads cuDNN through the dynamic linker, which does not look
# inside pip packages on its own.
os.environ["LD_LIBRARY_PATH"] = str(pathlib.Path(nvidia.cudnn.__file__).parent / "lib") + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("cuDNN path added")


## 3. Fetch the tool

Cloned rather than pasted so the notebook and the code cannot drift apart.

`REF` defaults to the default branch, which outlives any one pull request. While
the tool is still on a feature branch, set `REF` to that branch — and change it
back once it merges, because a branch deleted after merge takes every future run
of this notebook down with it.

In [ ]:
REPO = "psalovsky/agentmemory"
REF = "main"   # e.g. "claude/current-directory-yxo7oy" until that branch merges
SUBDIR = "STT_MultiLingva"   # "" once the tool lives in its own repository

!git clone --depth 1 -b "$REF" "https://github.com/$REPO" /content/repo

import pathlib, sys
TOOL = str(pathlib.Path("/content/repo") / SUBDIR)
if not pathlib.Path(TOOL, "app.py").is_file():
    raise SystemExit(f"The tool is not on {REPO}@{REF}. Set REF, or SUBDIR, to where it lives.")

# %cd changes the working directory but not where Python looks for modules, so
# `import app` in the next cell needs this.
if TOOL not in sys.path:
    sys.path.insert(0, TOOL)

%cd {TOOL}
!ls


## 4. Download the model once

Roughly 3 GB. Doing it here rather than on the first click means the interface
responds immediately instead of looking hung while it fetches.

In [ ]:
import app

MODEL = "large-v3"   # "large-v3-turbo" is much faster and weaker on Armenian

# Ask the app how it will choose, rather than assuming cuda/float16: the cache
# is keyed on that triple, so guessing wrong here warms an entry the interface
# never looks up and the first click downloads the model anyway.
DEVICE = "cuda" if app._cuda_available() else "cpu"
COMPUTE = "float16" if DEVICE == "cuda" else "int8"
if DEVICE == "cpu":
    print("No GPU visible. Runtime -> Change runtime type -> T4 GPU, then rerun from section 1.")

app.get_model(MODEL, DEVICE, COMPUTE)
print(f"{MODEL} loaded on {DEVICE} ({COMPUTE}) and cached")


## 5. The interface

Renders below this cell, and prints a `*.gradio.live` link that works from
another device — your phone, for instance — while this notebook keeps running.

Drop a file, pick the languages, press Transcribe. Audio and video both work:
PyAV pulls the audio stream out of an mp4, so a screen recording of a call needs
no conversion first.

**The share link is public to anyone holding it** for as long as the cell runs.
It is unlisted, not protected. Stop the cell when you are done.

In [ ]:
ui = app.build().queue()
ui.launch(share=True)


### Stopping it

Interrupting the cell (■) leaves the tunnel open until the runtime notices. This
closes it deliberately.

In [ ]:
ui.close()
print("interface stopped, share link dead")


## 6. Command line

For batch work, or when you want the exact flags in the output. Skip if section
5 did what you needed.

In [ ]:
from google.colab import files
import pathlib

uploaded = files.upload()
# files.upload() writes into the working directory, which section 3 moved into
# the clone -- so resolve against it rather than assuming a path.
AUDIO = str(pathlib.Path(next(iter(uploaded))).resolve())
UPLOADED = True   # this copy lives in the runtime and is ours to delete
print("using", AUDIO)


### Alternative: Drive

Instead of uploading. `UPLOADED = False` marks the file as yours rather than the
runtime's, so cleanup leaves it alone — deleting it would remove the original
recording from your Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# AUDIO = '/content/drive/MyDrive/meeting.m4a'
# UPLOADED = False


### Five minutes first

A short excerpt shows whether language detection is behaving before you commit
the whole file to a session Colab may reclaim mid-run.

In [ ]:
!ffmpeg -y -loglevel error -i "$AUDIO" -t 300 -ac 1 -ar 16000 /content/excerpt.wav
!python transcribe.py /content/excerpt.wav \
    --model large-v3 --device cuda --compute-type float16 \
    --languages ru,en --primary ru --out /content/out/excerpt


In [ ]:
import json, collections
lines = json.load(open('/content/out/excerpt.json'))
print(collections.Counter(l['language'] for l in lines))
print()
for l in lines:
    if l['language_probability'] < 0.75:
        print(f"{l['start']:7.1f}s  {l['language']}  p={l['language_probability']:.2f}  {l['text'][:70]}")


### The whole file

On a T4, `large-v3` runs roughly 20–30x real time: a four-hour recording lands
in about ten minutes. Keep the tab open — Colab disconnects idle sessions.

In [ ]:
import time
start = time.time()
!python transcribe.py "$AUDIO" \
    --model large-v3 --device cuda --compute-type float16 \
    --languages ru,en --primary ru --out /content/out/meeting
print(f"\n{(time.time() - start) / 60:.1f} minutes")


In [ ]:
from google.colab import files
for name in ['meeting.srt', 'meeting.txt', 'meeting.json']:
    files.download(f'/content/out/{name}')


### Clean up

Deleting the runtime's copy does not undo the upload — Google still received it.
This only limits how long it sits in the session. A file mounted from Drive is
never touched: it is the original, not a copy.

In [ ]:
import os

if UPLOADED:
    os.remove(AUDIO)
    print("removed the uploaded copy")
else:
    print(f"left {AUDIO} alone -- it is your Drive original, not a runtime copy")

if os.path.exists('/content/excerpt.wav'):
    os.remove('/content/excerpt.wav')
    print("removed the excerpt")


## Armenian

Add `hy` in the interface, or `--languages ru,en,hy` on the command line.
`large-v3` handles it out of the box at roughly 15.75% WER across dialects. If
Armenian is a large share of the recording, re-run the windows the JSON marks
`hy` through a fine-tuned model (`Chillarmo/whisper-large-v3-turbo-armenian`).

Note that `large-v3-turbo` is distilled and noticeably weaker on Armenian than
`large-v3`, so the speed trade is worse here than it looks for Russian and
English.